# Using LangChain Python Library with HuggingFace Locally
Examples for non-streaming and streaming calls using Hugging Face models locally.

## Setup

In [ ]:
# imports
import threading

from dotenv import load_dotenv
load_dotenv(override=True)

# LangChain and Hugging Face imports (third-party)
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
import torch

# local helper
from streaming_utils import get_chunk_content

# Hugging Face models expect a single plain-text prompt
def build_hf_prompt(messages):
    parts = []
    for msg in messages:
        content = msg.content if hasattr(msg, "content") else str(msg)
        content = content.strip()
        if content:
            parts.append(content)
    return "\n\n".join(parts) + "\n\nAnswer:"

# Hugging Face instanciation
def get_llm_instance(model, raw=False):
    if raw: # for the true streaming example, we need the raw model and tokenizer to use the TextIteratorStreamer
        # load raw transformers model + tokenizer and move to device
        tokenizer = AutoTokenizer.from_pretrained(model, use_fast=True)
        model_obj = AutoModelForCausalLM.from_pretrained(model)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model_obj.to(device)

        # Ensure pad token is set to eos to avoid attention/padding issues
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        if getattr(model_obj.config, 'pad_token_id', None) is None:
            model_obj.config.pad_token_id = model_obj.config.eos_token_id

        return model_obj, tokenizer, device
    
    else: # for the non-streaming example, we can use the HuggingFacePipeline wrapper which abstracts away the raw model and tokenizer and provides a simple interface for generation
        tokenizer = AutoTokenizer.from_pretrained(model, use_fast=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model_obj = AutoModelForCausalLM.from_pretrained(model)
        if getattr(model_obj.config, 'pad_token_id', None) is None:
            model_obj.config.pad_token_id = model_obj.config.eos_token_id
        hf_pipeline = pipeline("text-generation", model=model_obj, tokenizer=tokenizer, return_full_text=False)
        return HuggingFacePipeline(pipeline=hf_pipeline)


## ⚠️ Model Size Warning

**Important**: Running large language models locally requires significant resources. If you encounter out-of-memory errors, try a smaller model or run on a machine with more resources. 

**⚠️Monitor your system's resource usage during execution.**

**What Makes a Model Large or Small?**
Model size is primarily determined by the **number of parameters** (learnable weights) in the neural network:
- **Small models**: ~100M-500M parameters (e.g., 125M, 350M) → ~0.1-0.5 GB
- **Medium models**: ~1B-3B parameters (e.g., 1.3B, 2.7B) → ~1-3 GB  
- **Large models**: ~5B-10B parameters (e.g., 6B, 7B) → ~5-15 GB
- **Very Large models**: 10B+ parameters (e.g., 20B, 70B) → 20GB+

The GB size you see is the disk space needed to store these parameters. During inference, you'll need roughly 2x the model size in RAM/VRAM for comfortable operation.

**What to Monitor on Your Computer:**
- **RAM Usage**: Models load entirely into RAM. Monitor with Task Manager (Windows) or `htop` (Linux/Mac)
- **GPU Memory (VRAM)**: If using GPU acceleration, monitor VRAM usage with GPU monitoring tools
- **CPU Usage**: High CPU usage during model loading and inference
- **Disk Space**: Ensure enough space for model downloads (models can be 1GB-40GB+)
- **Network Speed**: Fast downloads are crucial for large models
- **Temperature**: GPUs/CPUs can heat up significantly under load
- **Power Consumption**: Large models can draw significant power

**Monitoring Tools:**
- **Windows**: Task Manager (Ctrl+Shift+Esc) → Performance tab
- **GPU**: MSI Afterburner, GPU-Z, or NVIDIA GeForce Experience
- **Command Line**: `nvidia-smi` for NVIDIA GPUs, `psutil` in Python scripts

**Resource Requirements:**
- At least 8 GB RAM recommended
- Compatible GPU (optional but recommended for better performance)
- Fast internet for initial download

In [ ]:
# System memory detection and model recommendations
import psutil

# Detect total system memory
total_memory = psutil.virtual_memory().total / (1024**3)  # Convert to GB
print(f"💻 Detected system memory: {total_memory:.1f} GB")

# Model options with approximate sizes (in GB)
models = [
    ("microsoft/DialoGPT-small", 0.1, "Small"),
    ("microsoft/DialoGPT-medium", 0.3, "Small"),
    ("distilgpt2", 0.3, "Small"),
    ("gpt2", 0.5, "Small"),
    ("gpt2-medium", 1.0, "Medium"),
    ("gpt2-large", 2.5, "Medium"),
    ("gpt2-xl", 5.0, "Large"),
    ("microsoft/DialoGPT-large", 1.5, "Medium"),
    ("facebook/opt-125m", 0.2, "Small"),
    ("facebook/opt-350m", 0.6, "Small"),
    ("facebook/opt-1.3b", 2.2, "Medium"),
    ("facebook/opt-2.7b", 4.8, "Large"),
    ("facebook/opt-6.7b", 11.5, "Very Large"),
    ("facebook/opt-13b", 23.0, "Very Large"),
]

print(f"📊 Available models: {len(models)} options across {len(set(cat for _, _, cat in models))} categories")

In [ ]:
# Get all acceptable models for your system
acceptable_models = [(name, size, cat) for name, size, cat in models if total_memory >= size * 1.5]

if acceptable_models:
    # Sort by size for consistent ordering
    acceptable_models.sort(key=lambda x: x[1])
    
    # Suggest a balanced middle option
    mid_index = len(acceptable_models) // 2
    middle_model = acceptable_models[mid_index]
    
    print(f"\n⚖️ Recommended balanced option: {middle_model[2]} model ({middle_model[0]})")
    print("   A good middle-ground choice balancing capability and resource usage.")
    
    print(f"\n📋 All models that should work on your system ({len(acceptable_models)} options):")
    for i, (name, size, cat) in enumerate(acceptable_models, 1):
        status = "⚖️" if (name, size, cat) == middle_model else "✅"
        print(f"   {status} {i}. {cat} model ({name}) - {size:.1f} GB")
else:
    print("\n⚠️ Your system may struggle with these models. Consider upgrading RAM or using cloud-based solutions.")

In [ ]:
# model
model_name = middle_model[0] # automatically selected based on your system memory.

# prepare message
system_message = "You are a helpful assistant."
user_message = "Tell me a joke about programming."

# LangChain uses its own message objects rather than simple dicts
messages = [SystemMessage(content=system_message), HumanMessage(content=user_message)]

## Non-Streaming

In [ ]:
def message(messages, model, **kwargs):
    llm_instance = get_llm_instance(model)
    prompt = build_hf_prompt(messages)

    used_kwargs = dict(kwargs)

    response = llm_instance.invoke(prompt, **used_kwargs)
    return response  # For HuggingFacePipeline, it's a string


# simple test call
print(f"Using model: {model_name}")
response = message(messages, model_name)
print(response)

## Simulated Streaming
Here we buffer chunks, not real-time token-by-token. 

In [ ]:
def simulated_stream_message(messages, model_name, **kwargs):
    llm_instance = get_llm_instance(model_name)
    full_response = ""
    prompt = build_hf_prompt(messages)
    hf_kwargs = {**kwargs}
    stream_iter = llm_instance.stream(prompt, **hf_kwargs)

    for chunk in stream_iter:
        content = get_chunk_content(chunk)
        print(content, end="", flush=True)
        full_response += content
    return full_response


# simple test call
print(f"Using model: {model_name}")
response = simulated_stream_message(messages, model_name)


## True Streaming
For true streaming, we use direct Transformers with TextIteratorStreamer.

In [ ]:
def stream_message(messages, model_name, **kwargs):
    """Generate text with true streaming from a local HF model."""

    # Use centralized helper to load model/tokenizer/device
    model, tokenizer, device = get_llm_instance(model_name, raw=True)

    # Prepare inputs (include attention_mask for reliable generation)
    prompt = build_hf_prompt(messages)
    inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Create streamer which yields decoded text chunks as they are generated
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    # Hard-coded generation kwargs with ability to override via kwargs
    used_kwargs = dict(
        pad_token_id=model.config.pad_token_id,
    )
    if kwargs:
        used_kwargs.update(kwargs)

    # Start generation in background thread so we can iterate the streamer in main thread
    thread = threading.Thread(
        target=model.generate,
        kwargs={
            'input_ids': inputs['input_ids'],
            'attention_mask': inputs.get('attention_mask'),
            'streamer': streamer,
            **used_kwargs,
        },
    )
    thread.start()

    # Collect and print chunks as they arrive
    full_response = ''
    try:
        for chunk in streamer:
            print(chunk, end='', flush=True)
            full_response += chunk
    finally:
        thread.join()

    return full_response


# simple test call
print(f"Using model: {model_name}")
response = stream_message(messages, model_name)